# FATE 피처 엔지니어링

MySQL에 적재된 주가와 KOSPI 데이터를 모델 학습용 데이터셋으로 변환합니다. 모든 피처는 해당 거래일 종가까지 알 수 있는 정보만 사용하고, 목표값은 다음 거래일 데이터로 생성합니다.

## 1. 환경 설정 및 데이터 불러오기

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.database import engine

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
stock_query = text('''
    SELECT s.ticker, s.name, p.trade_date, p.open_price, p.high_price,
           p.low_price, p.close_price, p.volume
    FROM stock_prices AS p
    JOIN stocks AS s ON s.stock_id = p.stock_id
    ORDER BY s.ticker, p.trade_date
''')

kospi_query = text('''
    SELECT miv.observation_date AS trade_date,
           miv.indicator_value AS kospi_close
    FROM market_indicator_values AS miv
    JOIN market_indicators AS mi ON mi.indicator_id = miv.indicator_id
    WHERE mi.indicator_code = 'KOSPI'
    ORDER BY miv.observation_date
''')

prices = pd.read_sql(stock_query, engine, parse_dates=['trade_date'])
kospi = pd.read_sql(kospi_query, engine, parse_dates=['trade_date'])

if prices.empty:
    raise ValueError('주가 데이터가 없습니다. 먼저 etl.stock_loader를 실행하세요.')
if kospi.empty:
    raise ValueError('KOSPI 데이터가 없습니다. 먼저 etl.market_loader를 실행하세요.')

prices.shape, kospi.shape

2026-08-24 23:22:05,353 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-08-24 23:22:05,355 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-08-24 23:22:05,358 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-08-24 23:22:05,359 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-08-24 23:22:05,361 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names
2026-08-24 23:22:05,362 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-08-24 23:22:05,367 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-08-24 23:22:05,368 INFO sqlalchemy.engine.Engine 
    SELECT s.ticker, s.name, p.trade_date, p.open_price, p.high_price,
           p.low_price, p.close_price, p.volume
    FROM stock_prices AS p
    JOIN stocks AS s ON s.stock_id = p.stock_id
    ORDER BY s.ticker, p.trade_date

2026-08-24 23:22:05,369 INFO sqlalchemy.engine.Engine [generated in 0.00235s] {}
2026-08-24 23:22:05,414 INFO sqlalchemy.engine.Engine ROLLBACK
2026-08-24 23:22:05,419 INFO sqlalchemy.engine.Engine BEGIN (implici

((771, 8), (257, 2))

## 2. 공통 시장 피처 생성

In [3]:
kospi = kospi.sort_values('trade_date').copy()
kospi['kospi_return_1d'] = kospi['kospi_close'].pct_change()
kospi['kospi_return_5d'] = kospi['kospi_close'].pct_change(5)
kospi['kospi_volatility_20d'] = kospi['kospi_return_1d'].rolling(20).std() * np.sqrt(252)

market_features = kospi[[
    'trade_date', 'kospi_return_1d', 'kospi_return_5d', 'kospi_volatility_20d'
]]
market_features.tail()

,trade_date,kospi_return_1d,kospi_return_5d,kospi_volatility_20d
252,2026-08-14,0.024159,0.114906,0.933922
253,2026-08-18,-0.015493,0.090508,0.919953
254,2026-08-19,-0.058031,0.019800,0.937448
255,2026-08-20,0.058940,0.041578,0.960764
256,2026-08-21,0.008810,0.014620,0.948437


## 3. 종목별 기술·거래 피처 생성

In [4]:
def add_stock_features(group: pd.DataFrame) -> pd.DataFrame:
    """한 종목의 과거 가격·거래량만 이용해 피처를 생성한다."""
    ticker = group.name
    group = group.sort_values('trade_date').copy()
    close = group['close_price']

    group['return_1d'] = close.pct_change()
    group['return_5d'] = close.pct_change(5)
    group['return_20d'] = close.pct_change(20)
    group['ma_5'] = close.rolling(5).mean()
    group['ma_20'] = close.rolling(20).mean()
    group['ma_60'] = close.rolling(60).mean()
    group['ma_5_ratio'] = close / group['ma_5'] - 1
    group['ma_20_ratio'] = close / group['ma_20'] - 1
    group['ma_60_ratio'] = close / group['ma_60'] - 1
    group['volatility_20d'] = group['return_1d'].rolling(20).std() * np.sqrt(252)
    group['volume_change_1d'] = group['volume'].pct_change()
    group['volume_ma_20'] = group['volume'].rolling(20).mean()
    group['volume_ratio_20'] = group['volume'] / group['volume_ma_20']

    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    group['rsi_14'] = 100 - (100 / (1 + rs))

    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    group['macd'] = ema_12 - ema_26
    group['macd_signal'] = group['macd'].ewm(span=9, adjust=False).mean()

    # 타깃: 다음 거래일의 수익률과 상승 여부. 마지막 행은 미래 정보가 없어 제외된다.
    group['target_return_1d'] = close.shift(-1) / close - 1
    group['target_up_1d'] = (group['target_return_1d'] > 0).astype('Int64')
    group.loc[group.index[-1], 'target_up_1d'] = pd.NA
    group['ticker'] = ticker
    return group

features = (
    prices.groupby('ticker', group_keys=False)
    .apply(add_stock_features, include_groups=False)
    .reset_index(drop=True)
)

# groupby 적용 과정에서 ticker가 인덱스로 남으므로, 원본 종목명은 ticker 기준으로 다시 연결한다.
stock_names = prices[['ticker', 'name']].drop_duplicates()
features = features.drop(columns=['name'], errors='ignore').merge(stock_names, on='ticker', how='left')
features = features.merge(market_features, on='trade_date', how='left')
features = features.sort_values(['ticker', 'trade_date']).reset_index(drop=True)
features.head()

,trade_date,open_price,high_price,low_price,close_price,volume,return_1d,return_5d,return_20d,ma_5,...,rsi_14,macd,macd_signal,target_return_1d,target_up_1d,ticker,name,kospi_return_1d,kospi_return_5d,kospi_volatility_20d
0,2025-08-01,266000.0,267000.0,257000.0,258000.0,3227394,NaN,NaN,NaN,NaN,...,NaN,0.000000,0.000000,0.000000,0,000660,SK하이닉스,NaN,NaN,NaN
1,2025-08-04,254500.0,260000.0,254000.0,258000.0,2281558,0.000000,NaN,NaN,NaN,...,NaN,0.000000,0.000000,0.021318,1,000660,SK하이닉스,0.009085,NaN,NaN
2,2025-08-05,264000.0,264500.0,260500.0,263500.0,2118171,0.021318,NaN,NaN,NaN,...,NaN,438.746439,87.749288,-0.018975,0,000660,SK하이닉스,0.015964,NaN,NaN
3,2025-08-06,260500.0,260500.0,257500.0,258500.0,1374559,-0.018975,NaN,NaN,NaN,...,NaN,378.633290,145.926088,0.013540,1,000660,SK하이닉스,0.000044,NaN,NaN
4,2025-08-07,258000.0,263000.0,250000.0,262000.0,2724438,0.013540,NaN,NaN,260000.0,...,NaN,606.423464,238.025563,-0.020992,0,000660,SK하이닉스,0.009237,NaN,NaN


## 4. 학습 데이터셋 확정

이동평균·변동성 계산에 필요한 초기 구간과 다음 날 가격이 없는 마지막 행을 제외합니다. `target_up_1d`는 분류 모델, `target_return_1d`는 회귀 모델의 목표값입니다.

In [5]:
feature_columns = [
    'return_1d', 'return_5d', 'return_20d',
    'ma_5_ratio', 'ma_20_ratio', 'ma_60_ratio',
    'volatility_20d', 'volume_change_1d', 'volume_ratio_20',
    'rsi_14', 'macd', 'macd_signal',
    'kospi_return_1d', 'kospi_return_5d', 'kospi_volatility_20d',
]
target_columns = ['target_return_1d', 'target_up_1d']

dataset = features.dropna(subset=feature_columns + target_columns).copy()
dataset['target_up_1d'] = dataset['target_up_1d'].astype(int)

dataset.groupby(['ticker', 'name']).agg(
    rows=('trade_date', 'size'),
    first_date=('trade_date', 'min'),
    last_date=('trade_date', 'max'),
    up_ratio=('target_up_1d', 'mean'),
)

,,rows,first_date,last_date,up_ratio
ticker,name,,,,
000660,SK하이닉스,197,2025-10-31,2026-08-20,0.568528
005930,삼성전자,197,2025-10-31,2026-08-20,0.558376
035420,NAVER,197,2025-10-31,2026-08-20,0.477157


## 5. 시간 순서로 학습·검증 데이터 분할

미래 정보가 학습에 섞이지 않도록 각 종목에서 앞 80%는 학습, 뒤 20%는 검증 데이터로 사용합니다.

In [6]:
def assign_split(group: pd.DataFrame, train_ratio: float = 0.8) -> pd.DataFrame:
    ticker = group.name
    group = group.sort_values('trade_date').copy()
    split_index = int(len(group) * train_ratio)
    group['split'] = 'train'
    group.iloc[split_index:, group.columns.get_loc('split')] = 'validation'
    group['ticker'] = ticker
    return group

dataset = (
    dataset.groupby('ticker', group_keys=False)
    .apply(assign_split, include_groups=False)
    .reset_index(drop=True)
)
dataset = dataset.drop(columns=['name'], errors='ignore').merge(stock_names, on='ticker', how='left')
dataset = dataset.sort_values(['ticker', 'trade_date']).reset_index(drop=True)
dataset.groupby(['ticker', 'split']).size().unstack(fill_value=0)

split,train,validation
ticker,,
000660,157,40
005930,157,40
035420,157,40


## 6. 저장

`fate_features.csv`는 목표값까지 갖춘 전체 학습 데이터셋이고, `fate_train.csv`와 `fate_validation.csv`는 모델 학습·평가에 사용합니다. `fate_prediction_features.csv`에는 다음 날 실제 값이 아직 없는 최신 거래일까지 포함해 예측에 사용합니다.

In [7]:
all_path = PROCESSED_DIR / 'fate_features.csv'
train_path = PROCESSED_DIR / 'fate_train.csv'
validation_path = PROCESSED_DIR / 'fate_validation.csv'
prediction_path = PROCESSED_DIR / 'fate_prediction_features.csv'

dataset.to_csv(all_path, index=False, encoding='utf-8-sig')
dataset.query("split == 'train'").to_csv(train_path, index=False, encoding='utf-8-sig')
dataset.query("split == 'validation'").to_csv(validation_path, index=False, encoding='utf-8-sig')

# target 값이 없는 최신 거래일도 포함한다. 예측 시에는 피처만 사용한다.
prediction_dataset = features.dropna(subset=feature_columns).copy()
prediction_dataset.to_csv(prediction_path, index=False, encoding='utf-8-sig')

print(f'예측 피처: {prediction_path} ({len(prediction_dataset):,}행)')
print(f'전체 데이터: {all_path} ({len(dataset):,}행)')
print(f'학습 데이터: {train_path} ({(dataset["split"] == "train").sum():,}행)')
print(f'검증 데이터: {validation_path} ({(dataset["split"] == "validation").sum():,}행)')

예측 피처: C:\Develops\fate\data\processed\fate_prediction_features.csv (594행)
전체 데이터: C:\Develops\fate\data\processed\fate_features.csv (591행)
학습 데이터: C:\Develops\fate\data\processed\fate_train.csv (471행)
검증 데이터: C:\Develops\fate\data\processed\fate_validation.csv (120행)


## 다음 단계

`ml/train.py`에서 `fate_train.csv`로 상승 여부 분류 모델을 학습하고, `fate_validation.csv`로 정확도·정밀도·재현율·ROC-AUC를 평가합니다.